# HeartAI — High-Performance Model Training Pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24abh/HeartAI/blob/main/notebooks/heart_disease_model_training.ipynb)

Trains and validates multiple clinical prediction algorithms with GPU acceleration in Google Colab:
- **Random Forest Classifier** (Scikit-Learn baseline)
- **Extreme Gradient Boosting** (`XGBClassifier`)
- **CardiacDeepNet** (PyTorch Neural Network)


In [ ]:
# Step 1: GPU Acceleration Check
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("NOTE: For accelerated deep learning, navigate to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")


In [ ]:
# Step 2: Ingest Data Splits
import os
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/HeartAI'
except ImportError:
    DRIVE_DIR = './drive_mock'

train_df = pd.read_csv(f"{DRIVE_DIR}/data/heart_train.csv")
val_df = pd.read_csv(f"{DRIVE_DIR}/data/heart_val.csv")

X_train, y_train = train_df.drop(columns=['target']), train_df['target']
X_val, y_val = val_df.drop(columns=['target']), val_df['target']

print(f"Training features shape: {X_train.shape}, Validation: {X_val.shape}")


In [ ]:
# Step 3: Train Scikit-Learn Random Forest & XGBoost
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# A. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict_proba(X_val)[:, 1]
print(f"[Random Forest] Val AUC: {roc_auc_score(y_val, rf_preds):.4f}, Accuracy: {accuracy_score(y_val, rf_preds >= 0.5):.4f}")

# B. XGBoost
xgb_model = XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.05, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict_proba(X_val)[:, 1]
print(f"[XGBoost]       Val AUC: {roc_auc_score(y_val, xgb_preds):.4f}, Accuracy: {accuracy_score(y_val, xgb_preds >= 0.5):.4f}")


In [ ]:
# Step 4: Train PyTorch CardiacDeepNet (Deep Neural Network)
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class CardiacDeepNet(nn.Module):
    def __init__(self, in_features=13):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.Mish(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.Mish(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

# Prepare PyTorch Tensors
X_tr_t = torch.tensor(X_train.values, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1).to(device)
X_val_t = torch.tensor(X_val.values, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1).to(device)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

model_pt = CardiacDeepNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model_pt.parameters(), lr=1e-3, weight_decay=1e-4)

# Training Loop
for epoch in range(1, 41):
    model_pt.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model_pt(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

model_pt.eval()
with torch.no_grad():
    val_logits = model_pt(X_val_t)
    val_probs = torch.sigmoid(val_logits).cpu().numpy().flatten()
    val_auc = roc_auc_score(y_val, val_probs)
    print(f"[CardiacDeepNet] Val AUC after 40 epochs: {val_auc:.4f}")
